# Steam 项目 EDA

目的：核对字段（A/B 切分）、清洗数据、摸底两个标签方案的不平衡度，为敲定最终标签与建模做准备。

泄漏红线：`positive_ratings` / `negative_ratings` / `owners` / `average_playtime` / `median_playtime` 只用于构造标签，绝不进入 X。

In [ ]:
import pathlib
import pandas as pd
import numpy as np

# 数据约定：把 steam.csv 放到本仓库的 data/ 目录下（data 不入库，见 README）
DATA_PATH = pathlib.Path.cwd() / "data" / "steam.csv"
if not DATA_PATH.exists():
    DATA_PATH = pathlib.Path("/Users/mac/Desktop/Machine Learning for Business/06_Datasets/steam/steam.csv")
    print("提示: 仓库内 data/steam.csv 不存在，改用本机绝对路径。")

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
df.info()

## 1. 字段核对（A/B 切分）

核对下面划分是否符合实际列名：
- **A 类（可作特征，发售时可得）**：`appid`(仅作 id), `name`(仅作 id), `release_date`, `english`, `developer`, `publisher`, `platforms`, `required_age`, `categories`, `genres`, `steamspy_tags`, `achievements`, `price`
- **B 类（仅用于标签）**：`positive_ratings`, `negative_ratings`, `owners`, `average_playtime`, `median_playtime`

In [ ]:
A_cols = ['appid', 'name', 'release_date', 'english', 'developer', 'publisher', 'platforms',
          'required_age', 'categories', 'genres', 'steamspy_tags', 'achievements', 'price']
B_cols = ['positive_ratings', 'negative_ratings', 'owners', 'average_playtime', 'median_playtime']

print('缺失列 A:', [c for c in A_cols if c not in df.columns])
print('缺失列 B:', [c for c in B_cols if c not in df.columns])
print('多余列:', [c for c in df.columns if c not in A_cols + B_cols])

## 2. 缺失值摸底

In [ ]:
missing = df.isnull().sum()
print(missing[missing > 0])
print('\n重复 appid:', df['appid'].duplicated().sum())
print('重复 name:', df['name'].duplicated().sum())

## 3. owners 区间 → 转中点（仅用于标签 B）

In [ ]:
df['owners'].value_counts()

In [ ]:
# "0-0" 区间中点取 0；形如 "100000-200000" 取 (100000+200000)/2
def owners_midpoint(s):
    lo, hi = s.split('-')
    lo, hi = int(lo), int(hi)
    return 0 if hi == 0 else (lo + hi) / 2

df['owners_mid'] = df['owners'].apply(owners_midpoint)
print(df['owners_mid'].describe())

In [ ]:
# 好评率（用于标签 A）
total = df['positive_ratings'] + df['negative_ratings']
df['pos_rate'] = np.where(total > 0, df['positive_ratings'] / total, np.nan)
print('有评价的游戏占比:', (total > 0).mean())
print('无评价(好评率NaN)游戏数:', df['pos_rate'].isna().sum())

# 发行年份
df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year
print('\nrelease_year 缺失:', df['release_year'].isna().sum())
df['release_year'].value_counts().sort_index().plot(kind='bar', figsize=(14, 4), title='游戏数量 by 发行年份');

## 4. 标签方案不平衡度摸底

先看整体，再看 2015–2019（价格/属性更接近发行当年，见提案 §6 局限）。

In [ ]:
def label_stats(d):
    n = len(d)
    # 方案 A：好评率 >= 0.85
    a = d['pos_rate'].dropna()
    hit_a = (a >= 0.85).mean()
    # 方案 B：按 release_year 分组，组内 owners_mid 排名前 10%
    g = d.dropna(subset=['owners_mid', 'release_year']).copy()
    g['rank'] = g.groupby('release_year')['owners_mid'].rank(pct=True, ascending=True)
    hit_b = (g['rank'] >= 0.90).mean()
    print(f'n={n} | A(好评率>=0.85): 可用 {a.notna().sum()}, 命中率 {hit_a:.1%} | '
          f'B(同年代owners前10%): 可用 {len(g)}, 命中率 {hit_b:.1%}')
    return {'n': n, 'A_rate': hit_a, 'B_rate': hit_b, 'A_n': int(a.notna().sum()), 'B_n': len(g)}

print('— 全样本 —')
s_all = label_stats(df)
print('— 2015–2019 —')
s_recent = label_stats(df[df['release_year'].between(2015, 2019)])

## 5. 记录结论

根据上面结果记录：
- 最终标签选 A / B / 两者都做？
- 无评价游戏怎么处理？
- 样本范围是否限定年份？
- owners 区间解析有没有遇到怪值？